# Week 5: Clean Listing Dataset

The Week 4-5 cleaning so far covered the **sold** dataset. This notebook cleans
the **listing** dataset so the Tableau "New Listings" dashboard can count new
listings per month. Listings are simpler than solds (no sale timeline, no price
ratios), but the combined file has duplicate columns and a few duplicate rows
to resolve first.

In [ ]:
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

INPUT_FILE = (
    PROCESSED_DIR
    / "crmls_listings_combined_residential_with_mortgage_rates_202401_202606.csv"
)
listings = pd.read_csv(INPUT_FILE, low_memory=False)
print(f"{listings.shape[0]:,} rows x {listings.shape[1]} columns")

## 1. Drop duplicate columns

The combine step left 11 columns duplicated with a `.1` suffix (pandas renames
same-named columns on read). Each `.1` column is identical to its original, so
they are pure redundancy and get dropped.

In [ ]:
# Duplicate columns carry a ".1" suffix; confirm each matches its original,
# then drop the ".1" copies.
dot1_cols = [c for c in listings.columns if c.endswith(".1")]
for c in dot1_cols:
    original = c[:-2]
    identical = listings[original].equals(listings[c])
    print(f"{c:28s} == {original:20s} identical: {identical}")

listings = listings.drop(columns=dot1_cols)
print(f"\nDropped {len(dot1_cols)} duplicate columns -> {listings.shape[1]} columns")

## 2. Drop duplicate rows

Each listing should appear once. A few ListingKeys repeat with conflicting
listing dates; keep the earliest ListingContractDate (the first time the
property came to market) so "new listings" counts each home once.

In [ ]:
listings["ListingContractDate"] = pd.to_datetime(
    listings["ListingContractDate"], errors="coerce"
)

dup = listings["ListingKey"].duplicated(keep=False)
print(f"Rows with a duplicated ListingKey: {int(dup.sum()):,}")

rows_before = len(listings)
listings = (
    listings.sort_values("ListingContractDate")
    .drop_duplicates(subset="ListingKey", keep="first")
    .reset_index(drop=True)
)
print(f"After dedupe: {rows_before:,} -> {len(listings):,} "
      f"(one row per ListingKey)")

## 3. Filter to California

Same bounding-box logic used for the sold data: keep explicit CA rows, plus
missing-state rows whose coordinates fall inside California.

In [ ]:
state = listings["StateOrProvince"].astype("string").str.strip().str.upper()
missing_state = state.isna() | state.eq("")

lat = pd.to_numeric(listings["Latitude"], errors="coerce")
lon = pd.to_numeric(listings["Longitude"], errors="coerce")
plausible_ca = lat.between(32, 42, inclusive="both") & lon.between(-125, -114, inclusive="both")

keep = state.isin(["CA", "CALIFORNIA"]) | (missing_state & plausible_ca)
removed = int((~keep).sum())
listings = listings.loc[keep].copy()
print(f"State filter: removed {removed} non-CA rows -> {len(listings):,}")

## 4. Fix coordinates

Zero coordinates are placeholders (set to missing); positive longitudes are
dropped minus signs (flip when the result lands in California, else missing).
Counts are tiny here, but the fix keeps the map/heatmap dashboards clean.

In [ ]:
listings["Latitude"] = pd.to_numeric(listings["Latitude"], errors="coerce")
listings["Longitude"] = pd.to_numeric(listings["Longitude"], errors="coerce")

zero = listings["Latitude"].eq(0) | listings["Longitude"].eq(0)
listings.loc[zero, ["Latitude", "Longitude"]] = pd.NA
print(f"Zero coordinates set to missing: {int(zero.sum())}")

positive = listings["Longitude"].gt(0).fillna(False)
flippable = (
    positive
    & listings["Latitude"].between(32, 42)
    & (-listings["Longitude"]).between(-125, -114)
)
listings.loc[flippable, "Longitude"] = -listings.loc[flippable, "Longitude"]
listings.loc[positive & ~flippable, ["Latitude", "Longitude"]] = pd.NA
print(f"Positive longitude: {int(flippable.sum())} flipped, "
      f"{int((positive & ~flippable).sum())} set to missing")

## 5. Drop high-missing columns and add the month key

Drop columns that are more than 90% empty (same rule as the sold cleaning —
this also clears the all-empty columns seen earlier), then derive `list_yrmo`
from ListingContractDate for the monthly "new listings" axis.

In [ ]:
missing_pct = listings.isna().mean() * 100
high_missing = missing_pct[missing_pct > 90].index.tolist()
listings = listings.drop(columns=high_missing)
print(f"Dropped {len(high_missing)} columns >90% missing")

# Monthly key for the "new listings per month" dashboard.
listings["list_yrmo"] = listings["ListingContractDate"].dt.strftime("%Y-%m")
print(f"\nDate range: {listings['list_yrmo'].min()} -> {listings['list_yrmo'].max()}")
print(f"Months: {listings['list_yrmo'].nunique()}")
print(f"Final shape: {listings.shape[0]:,} rows x {listings.shape[1]} columns")

## 6. Quick sanity check: new listings per month

In [ ]:
new_by_month = (
    listings.groupby("list_yrmo").size().rename("new_listings").reset_index()
)
display(new_by_month)

## 7. Save the cleaned listing dataset

In [ ]:
OUTPUT_FILE = PROCESSED_DIR / "crmls_listings_final_202401_202606.csv"
listings.to_csv(OUTPUT_FILE, index=False)
print(f"Saved: {OUTPUT_FILE}")
print(f"  {listings.shape[0]:,} rows x {listings.shape[1]} columns")